# 07. Model Audit and Improvement (Clipped Predictions)

**Tác giả / Nhóm thực hiện:** Real Estate Data Science Team  
**Mục đích:** 
Kiểm định chuyên sâu các mô hình (TP.HCM và Hà Nội) về hiện tượng dự báo ra số âm. 
Giải pháp: Áp dụng hàm `np.maximum(y_pred, 0)` (Clipping) để đảm bảo tính logic kinh tế (giá nhà không thể âm), sau đó so sánh với đầu ra gốc.


In [1]:
import pandas as pd
import numpy as np
import joblib
from sklearn.model_selection import train_test_split
from sklearn.metrics import r2_score, mean_absolute_error, mean_squared_error

df_feat = pd.read_csv('../data/processed/housing_features.csv')


## PHẦN 1: KIỂM ĐỊNH MÔ HÌNH TP. HỒ CHÍ MINH

In [2]:
df_hcm = df_feat[df_feat['province'] == 'Hồ Chí Minh'].copy()
X_hcm = df_hcm.drop(columns=['price_million_vnd'])
y_hcm = df_hcm['price_million_vnd']
X_train_hcm, X_test_hcm, y_train_hcm, y_test_hcm = train_test_split(X_hcm, y_hcm, test_size=0.2, random_state=42)

model_hcm = joblib.load('../models/linear_regression_hcm.pkl')
raw_preds_hcm = model_hcm.predict(X_test_hcm)
clipped_preds_hcm = np.maximum(raw_preds_hcm, 0)

num_negs_hcm = (raw_preds_hcm < 0).sum()

print("=== KIỂM ĐỊNH HIỆN TƯỢNG GIÁ ÂM (TP.HCM) ===")
print(f"Tổng số dự đoán trong Test Set: {len(y_test_hcm)}")
print(f"Số lượng dự đoán < 0 (trước khi xử lý): {num_negs_hcm} ({num_negs_hcm/len(raw_preds_hcm)*100:.2f}%)")
print(f"Giá trị âm nhỏ nhất: {raw_preds_hcm.min():,.2f} triệu VNĐ")

print("\n=== SO SÁNH HIỆU NĂNG SAU KHI XỬ LÝ (CLIPPING) ===")
print(f"R² Score (Raw):      {r2_score(y_test_hcm, raw_preds_hcm):.4f}")
print(f"R² Score (Clipped):  {r2_score(y_test_hcm, clipped_preds_hcm):.4f}")
print(f"MAE (Raw):           {mean_absolute_error(y_test_hcm, raw_preds_hcm):,.2f}")
print(f"MAE (Clipped):       {mean_absolute_error(y_test_hcm, clipped_preds_hcm):,.2f}")


=== KIỂM ĐỊNH HIỆN TƯỢNG GIÁ ÂM (TP.HCM) ===
Tổng số dự đoán trong Test Set: 3821
Số lượng dự đoán < 0 (trước khi xử lý): 4 (0.10%)
Giá trị âm nhỏ nhất: -40,913.22 triệu VNĐ

=== SO SÁNH HIỆU NĂNG SAU KHI XỬ LÝ (CLIPPING) ===
R² Score (Raw):      0.5343
R² Score (Clipped):  0.5345
MAE (Raw):           29,929.71
MAE (Clipped):       29,913.56


## PHẦN 2: KIỂM ĐỊNH MÔ HÌNH HÀ NỘI

In [3]:
df_hn = df_feat[df_feat['province'] == 'Hà Nội'].copy()
X_hn = df_hn.drop(columns=['price_million_vnd'])
y_hn = df_hn['price_million_vnd']
X_train_hn, X_test_hn, y_train_hn, y_test_hn = train_test_split(X_hn, y_hn, test_size=0.2, random_state=42)

model_hn = joblib.load('../models/linear_regression_hanoi.pkl')
raw_preds_hn = model_hn.predict(X_test_hn)
clipped_preds_hn = np.maximum(raw_preds_hn, 0)

num_negs_hn = (raw_preds_hn < 0).sum()

print("=== KIỂM ĐỊNH HIỆN TƯỢNG GIÁ ÂM (HÀ NỘI) ===")
print(f"Tổng số dự đoán trong Test Set: {len(y_test_hn)}")
print(f"Số lượng dự đoán < 0 (trước khi xử lý): {num_negs_hn} ({num_negs_hn/len(raw_preds_hn)*100:.2f}%)")
print(f"Giá trị âm nhỏ nhất: {raw_preds_hn.min():,.2f} triệu VNĐ")

print("\n=== SO SÁNH HIỆU NĂNG SAU KHI XỬ LÝ (CLIPPING) ===")
print(f"R² Score (Raw):      {r2_score(y_test_hn, raw_preds_hn):.4f}")
print(f"R² Score (Clipped):  {r2_score(y_test_hn, clipped_preds_hn):.4f}")
print(f"MAE (Raw):           {mean_absolute_error(y_test_hn, raw_preds_hn):,.2f}")
print(f"MAE (Clipped):       {mean_absolute_error(y_test_hn, clipped_preds_hn):,.2f}")


=== KIỂM ĐỊNH HIỆN TƯỢNG GIÁ ÂM (HÀ NỘI) ===
Tổng số dự đoán trong Test Set: 3738
Số lượng dự đoán < 0 (trước khi xử lý): 4 (0.11%)
Giá trị âm nhỏ nhất: -16,465.39 triệu VNĐ

=== SO SÁNH HIỆU NĂNG SAU KHI XỬ LÝ (CLIPPING) ===
R² Score (Raw):      0.4710
R² Score (Clipped):  0.4711
MAE (Raw):           40,256.64
MAE (Clipped):       40,243.11
